In [1]:
!pip -q install datasets pandas tqdm tokenizers

In [4]:
import os
import json
import random
import pandas as pd

from tqdm.auto import tqdm
from datasets import load_dataset
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(
    "/kaggle/input/datasets/punitkashyap2007/virgo-tokenizer/virgo_tokenizer.json"
)

OUTPUT_FILE = "information_extraction.csv"

existing_pairs = set()
rows = []

In [5]:
def clean_text(text):
    text = str(text)
    text = text.replace("\u200b", " ")
    text = text.replace("\xa0", " ")
    text = " ".join(text.split())
    return text.strip()


def token_count(text):
    return len(tokenizer.encode(text).ids)


def add_sample(prompt, response):
    prompt = clean_text(prompt)
    response = clean_text(response)

    if not prompt or not response:
        return

    if len(prompt) < 8 or len(response) < 2:
        return

    if token_count(prompt) > 1024:
        return

    if token_count(response) > 1024:
        return

    key = (prompt, response)

    if key in existing_pairs:
        return

    existing_pairs.add(key)

    rows.append({
        "prompt": prompt,
        "response": response
    })

In [7]:
dataset = load_dataset(
    "lhoestq/conll2003",
    split="train"
)

print(f"Samples: {len(dataset):,}")
dataset

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/281k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

Samples: 14,041


Dataset({
    features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
    num_rows: 14041
})

In [8]:
ner_prompts = [
    "Extract all named entities from the following text.",
    "Identify every named entity mentioned below.",
    "Find all named entities in the passage.",
    "Recognize all named entities.",
    "List every named entity appearing in the text.",
    "Detect all named entities from the sentence.",
    "Identify people, organizations, and locations.",
    "Extract all people, organizations, locations, and miscellaneous entities.",
    "Return every entity together with its category.",
    "Find all entities and classify them.",
    "Recognize entity mentions in the text.",
    "Extract all proper named entities.",
    "Identify all proper nouns representing entities.",
    "Find all entity names.",
    "Locate every named entity.",
    "Extract all entity mentions from the passage.",
    "Identify every person mentioned.",
    "Identify every organization mentioned.",
    "Identify every location mentioned.",
    "Identify every miscellaneous entity mentioned.",
    "List all people found in the text.",
    "List all organizations found in the text.",
    "List all locations found in the text.",
    "Return every recognized entity.",
    "Extract named entities while preserving their types.",
    "Identify entities and return them grouped by category.",
    "Return all entities in JSON format.",
    "Produce a structured list of all entities.",
    "Extract every unique entity.",
    "Return unique entity mentions only.",
    "Identify repeated entity mentions.",
    "Extract entities without duplicates.",
    "Find every entity occurrence.",
    "Find every entity appearing in the paragraph.",
    "Recognize all entity spans.",
    "Detect every named object referenced.",
    "Identify all real-world entities.",
    "Find all references to people and places.",
    "Extract references to organizations and places.",
    "Extract all proper names.",
    "Identify important entities in the passage.",
    "Extract entities from the document.",
    "Identify entity boundaries.",
    "Recognize all NER labels in the text.",
    "Perform named entity recognition on the following text.",
    "Run NER on the following passage.",
    "Annotate the following text with named entities.",
    "Detect named entities and classify them.",
    "Tag all named entities.",
    "Return entity annotations.",
    "Identify every entity and assign its type.",
    "Extract all entities and their labels.",
    "Find entities together with their categories.",
    "Return entities grouped as PERSON, ORGANIZATION, LOCATION, and MISCELLANEOUS.",
    "Extract entity information.",
    "Generate a list of named entities.",
    "Identify all names appearing below.",
    "Find all identifiable entities.",
    "Recognize all important names.",
    "Extract all entity references.",
    "List all recognized names.",
    "Detect every proper noun that is a named entity.",
    "Return every organization, person, and location.",
    "Find all geopolitical entities.",
    "Extract all cities, countries, and organizations.",
    "Identify every company name.",
    "Extract institution names.",
    "Recognize place names.",
    "Find all human names.",
    "Extract all geographical names.",
    "List every organization appearing in the document.",
    "Extract all entities into JSON.",
    "Represent named entities as structured JSON.",
    "Produce JSON containing entity types.",
    "Return entity extraction results in JSON format.",
    "Generate a dictionary of extracted entities.",
    "Return entity categories with their values.",
    "Extract entities for downstream processing.",
    "Identify entities for information extraction.",
    "Recognize entities suitable for indexing.",
    "Extract semantic entities.",
    "Identify all named references.",
    "Extract entity labels from the text.",
    "Return the entity inventory.",
    "Perform entity extraction.",
    "Identify all entity instances.",
    "Extract every categorized entity.",
    "Find entity mentions and their classes.",
    "Produce entity annotations in structured format.",
    "Identify all PERSON entities.",
    "Identify all ORGANIZATION entities.",
    "Identify all LOCATION entities.",
    "Identify all MISCELLANEOUS entities.",
    "Recognize all entities and organize them by label.",
    "Extract all classified entities.",
    "Generate an entity report.",
    "Return all discovered named entities.",
    "Find all entities present in this document.",
    "Extract every identifiable named object.",
    "Perform entity recognition and classification.",
    "Generate the complete entity list.",
    "List all categorized entity mentions.",
    "Identify every named concept belonging to NER categories.",
    "Return entity extraction output.",
    "Extract structured named entity information."
]

In [10]:
label_map = {
    "PER": "PERSON",
    "ORG": "ORGANIZATION",
    "LOC": "LOCATION",
    "MISC": "MISCELLANEOUS"
}

In [14]:
start = len(rows)

response_styles = [
    "json",
    "bullet",
    "markdown"
]

id2label = {
    0: "O",
    1: "B-PER",
    2: "I-PER",
    3: "B-ORG",
    4: "I-ORG",
    5: "B-LOC",
    6: "I-LOC",
    7: "B-MISC",
    8: "I-MISC",
}

label_map = {
    "PER": "PERSON",
    "ORG": "ORGANIZATION",
    "LOC": "LOCATION",
    "MISC": "MISCELLANEOUS",
}

for sample in tqdm(dataset):

    tokens = sample["tokens"]
    tags = sample["ner_tags"]

    entities = {
        "PERSON": [],
        "ORGANIZATION": [],
        "LOCATION": [],
        "MISCELLANEOUS": []
    }

    current_tokens = []
    current_type = None

    for token, tag_id in zip(tokens, tags):

        tag = id2label[tag_id]

        if tag == "O":
            if current_tokens:
                entity = " ".join(current_tokens)
                if entity not in entities[current_type]:
                    entities[current_type].append(entity)
            current_tokens = []
            current_type = None
            continue

        prefix, label = tag.split("-", 1)
        label = label_map[label]

        if prefix == "B":
            if current_tokens:
                entity = " ".join(current_tokens)
                if entity not in entities[current_type]:
                    entities[current_type].append(entity)
            current_tokens = [token]
            current_type = label

        elif prefix == "I":
            if current_tokens:
                current_tokens.append(token)
            else:
                current_tokens = [token]
                current_type = label

    if current_tokens:
        entity = " ".join(current_tokens)
        if entity not in entities[current_type]:
            entities[current_type].append(entity)

    text = " ".join(tokens)

    style = random.choice(response_styles)

    if style == "json":
        response = json.dumps(
            {k: v for k, v in entities.items() if v},
            ensure_ascii=False,
            indent=2
        )

    elif style == "bullet":
        lines = []
        for k, v in entities.items():
            if v:
                lines.append(f"{k}:")
                lines.extend(f"- {item}" for item in v)
        response = "\n".join(lines)

    else:
        table = ["| Type | Entity |", "|---|---|"]
        for k, v in entities.items():
            for item in v:
                table.append(f"| {k} | {item} |")
        response = "\n".join(table)

    add_sample(
        f"{random.choice(ner_prompts)}\n\n{text}",
        response
    )

print(f"Added: {len(rows) - start:,}")

  0%|          | 0/14041 [00:00<?, ?it/s]

Added: 13,085


In [15]:
df = pd.DataFrame(rows)
df.to_csv(OUTPUT_FILE, index=False)

print(f"Saved {len(df):,} samples to {OUTPUT_FILE}")

Saved 13,085 samples to information_extraction.csv


In [16]:
df = pd.DataFrame(rows)

for i in range(5):
    print("=" * 100)
    print(f"Sample {i+1}")
    print("=" * 100)
    print("PROMPT:")
    print(df.iloc[i]["prompt"])
    print("\nRESPONSE:")
    print(df.iloc[i]["response"])
    print()

Sample 1
PROMPT:
Extract every identifiable named object. EU rejects German call to boycott British lamb .

RESPONSE:
| Type | Entity | |---|---| | ORGANIZATION | EU | | MISCELLANEOUS | German | | MISCELLANEOUS | British |

Sample 2
PROMPT:
Extract named entities while preserving their types. Peter Blackburn

RESPONSE:
PERSON: - Peter Blackburn

Sample 3
PROMPT:
Identify every location mentioned. BRUSSELS 1996-08-22

RESPONSE:
{ "LOCATION": [ "BRUSSELS" ] }

Sample 4
PROMPT:
Identify important entities in the passage. The European Commission said on Thursday it disagreed with German advice to consumers to shun British lamb until scientists determine whether mad cow disease can be transmitted to sheep .

RESPONSE:
| Type | Entity | |---|---| | ORGANIZATION | European Commission | | MISCELLANEOUS | German | | MISCELLANEOUS | British |

Sample 5
PROMPT:
Recognize all entities and organize them by label. Germany 's representative to the European Union 's veterinary committee Werner Zwingma

In [17]:
def clean_text(text):
    text = str(text).replace("\r", "")
    text = "\n".join(line.rstrip() for line in text.splitlines())
    return text.strip()


def token_count(text):
    return len(tokenizer.encode(text).ids)


def add_sample(prompt, response):
    prompt = clean_text(prompt)
    response = clean_text(response)

    if not prompt or not response:
        return

    pair = (prompt, response)
    if pair in existing_pairs:
        return

    if token_count(prompt) > 1024:
        return

    if token_count(response) > 1024:
        return

    if token_count(prompt + "\n" + response) > 1024:
        return

    existing_pairs.add(pair)
    rows.append({
        "prompt": prompt,
        "response": response
    })

In [18]:
dataset = load_dataset(
    "rajpurkar/squad_v2",
    split="train"
)

print(f"Samples: {len(dataset):,}")
dataset

README.md: 0.00B [00:00, ?B/s]

squad_v2/train-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

squad_v2/validation-00000-of-00001.parqu(…):   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

Samples: 130,319


Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 130319
})

In [19]:
qa_prompts = [
    "Read the passage and answer the question.",
    "Answer the question using only the provided passage.",
    "Extract the correct answer from the passage.",
    "Find the answer in the given context.",
    "Use the passage to answer the question.",
    "Read the context carefully and respond to the question.",
    "Locate the answer in the passage.",
    "Answer based only on the information provided.",
    "Identify the correct answer from the context.",
    "Read the article and answer the question.",
    "Determine the answer using the passage.",
    "Find the information requested in the text.",
    "Respond using evidence from the passage.",
    "Extract the requested information from the context.",
    "Use the given paragraph to answer the question.",
    "Provide the answer found in the passage.",
    "Read the context and identify the answer.",
    "Return the exact answer from the passage.",
    "Identify the span that answers the question.",
    "Answer the question without using outside knowledge.",
    "Read the following passage and answer.",
    "Carefully read the passage before answering.",
    "Answer the question from the given context only.",
    "Determine the correct response from the passage.",
    "Extract the answer exactly as written.",
    "Identify the relevant text span.",
    "Answer using only information in the context.",
    "Use the context below to answer.",
    "Read and answer accurately.",
    "Find the evidence in the passage.",
    "Answer the following comprehension question.",
    "Read the document and answer the question.",
    "Use the article to answer correctly.",
    "Answer according to the passage.",
    "Provide a concise answer using the context.",
    "Identify the answer supported by the passage.",
    "Locate the relevant information.",
    "Return the answer based on the context.",
    "Find the answer mentioned in the text.",
    "Answer the question from the passage below.",
    "Read the context and extract the answer.",
    "Identify the best answer using the text.",
    "Answer using only the supplied passage.",
    "Determine the answer from the information given.",
    "Extract the correct response.",
    "Use textual evidence to answer the question.",
    "Find the exact phrase answering the question.",
    "Read carefully and answer precisely.",
    "Provide the correct answer from the passage.",
    "Answer with the information present in the context.",
    "Solve the reading comprehension task.",
    "Extract the answer span.",
    "Read the context before answering.",
    "Return the answer exactly if possible.",
    "Identify the sentence containing the answer.",
    "Find and return the answer.",
    "Answer the comprehension question accurately.",
    "Use only the provided context.",
    "Answer without making assumptions.",
    "Read and extract the requested fact.",
    "Determine the factual answer.",
    "Read the passage below and answer.",
    "Answer the question using textual evidence.",
    "Locate the answer in the article.",
    "Find the correct answer from the context.",
    "Provide the answer supported by the passage.",
    "Read the paragraph and answer.",
    "Identify the answer from the document.",
    "Extract the relevant fact.",
    "Answer from the given article.",
    "Read the text and identify the answer.",
    "Answer the question as stated in the passage.",
    "Determine the answer directly from the context.",
    "Extract the information requested.",
    "Answer using the accompanying passage.",
    "Find the supporting answer in the context.",
    "Read before answering the question.",
    "Return only the answer supported by the passage.",
    "Use the context to identify the correct answer.",
    "Read the passage thoroughly and answer.",
    "Answer using information explicitly stated.",
    "Identify the answer without outside knowledge.",
    "Read the context and respond.",
    "Find the exact information requested.",
    "Extract the answer from the following article.",
    "Answer accurately from the provided text.",
    "Read the evidence and answer.",
    "Use the context as your only source.",
    "Answer by extracting the relevant span.",
    "Determine the answer mentioned in the passage.",
    "Locate and extract the answer.",
    "Read the context and solve the question.",
    "Answer using only facts from the passage.",
    "Identify the correct response from the article.",
    "Provide the answer exactly as supported.",
    "Extract the factual answer.",
    "Read the passage and identify the correct answer.",
    "Find the requested information in the passage.",
    "Answer the question based entirely on the context.",
    "Read and answer the reading comprehension question."
]

In [20]:
start = len(rows)

answerable_styles = [
    lambda x: x,
    lambda x: f"The answer is {x}.",
    lambda x: f"{x}",
    lambda x: f"Answer: {x}",
    lambda x: f"The correct answer is {x}.",
]

unanswerable_responses = [
    "Cannot be determined from the passage.",
    "The passage does not contain enough information.",
    "No answer is provided in the context.",
    "The answer cannot be found in the passage.",
    "Insufficient information.",
    "Not mentioned in the passage.",
    "Cannot answer based on the provided context.",
    "The context does not specify the answer.",
    "There is no answer in the passage.",
    "Unknown based on the given context."
]

for sample in tqdm(dataset):

    context = clean_text(sample["context"])
    question = clean_text(sample["question"])

    if not context or not question:
        continue

    prompt = (
        f"{random.choice(qa_prompts)}\n\n"
        f"Passage:\n{context}\n\n"
        f"Question:\n{question}"
    )

    answers = sample["answers"]["text"]

    if len(answers):
        answer = clean_text(random.choice(answers))
        response = random.choice(answerable_styles)(answer)
    else:
        response = random.choice(unanswerable_responses)

    add_sample(prompt, response)

print(f"Added: {len(rows)-start:,}")

  0%|          | 0/130319 [00:00<?, ?it/s]

Added: 130,319


In [21]:
df = pd.DataFrame(rows)
df.to_csv(OUTPUT_FILE, index=False)

print(f"Saved {len(df):,} samples to {OUTPUT_FILE}")

Saved 143,404 samples to information_extraction.csv
